In [ ]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore') 

In [ ]:
X_train = pd.read_csv('../Datos/preparados/TrainX.csv')
y_train = pd.read_csv('../Datos/preparados/TrainY.csv')
X_val = pd.read_csv('../Datos/preparados/ValidationX.csv') 
y_val = pd.read_csv('../Datos/preparados/ValidationY.csv')
X_test = pd.read_csv('../Datos/preparados/TestX.csv')
y_test = pd.read_csv('../Datos/preparados/TestY.csv')

y_train = y_train.values.ravel()
y_val = y_val.values.ravel()
y_test = y_test.values.ravel()

LDA es un modelo de clasificación y no de regresion, asi que para poder usar este modelo debo pasar roi a categorias que es lo que hago enl a siguiente parte

In [ ]:
p33, p66 = np.percentile(y_train, [33, 66])

def convertir_a_clases_con_cortes(y, cortes):
    return np.digitize(y, bins=cortes)

cortes = [p33, p66]
y_train_class = convertir_a_clases_con_cortes(y_train, cortes)
y_val_class = convertir_a_clases_con_cortes(y_val, cortes)
y_test_class = convertir_a_clases_con_cortes(y_test, cortes)

print("Cortes (33%,66%) calculados desde TRAIN:", cortes)
print("Distribución clases (train/val/test):",
      np.bincount(y_train_class), np.bincount(y_val_class), np.bincount(y_test_class))

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
resultados = []
#ciclos anidados para probar combinaciones de hiperparametros
for solver in ['svd', 'lsqr', 'eigen']:
    for shrink in [None, 'auto']:
        for tol in [0.0001, 0.001, 0.01]:
            try:
                lda = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrink, tol=tol)#la iteracion en la que va
                lda.fit(X_train, y_train_class)#entrenamiento con los datoss
                y_pred_train = lda.predict(X_train)#prediccion de los datos de prediccion
                y_pred_val = lda.predict(X_val)#prediccion con datos de validacion

                error_train = 1 - accuracy_score(y_train_class, y_pred_train)
                error_val = 1 - accuracy_score(y_val_class, y_pred_val)

                resultados.append({ #al arreglo de resultados se pone la iteracion actuakl
                    'solver': solver,
                    'shrinkage': shrink,
                    'tol': tol,
                    'error_train': error_train,
                    'error_val': error_val
                })
            except Exception as e:
                print(f"Error con solver={solver}, shrink={shrink}, tol={tol}: {e}")

In [ ]:
df_resultados = pd.DataFrame(resultados) #crea un dataframe con los resultados
df_resultados.sort_values(by='error_val', inplace=True)
df_resultados.reset_index(drop=True, inplace=True)
df_resultados

In [ ]:
#el mejor parametro es el que quedo primero al organizarlo
mejor = df_resultados.iloc[0]
print("Mejores hiperparámetros encontrados:")
print(mejor)

In [ ]:
lda_best = LinearDiscriminantAnalysis( 
    solver=mejor['solver'],
    shrinkage=mejor['shrinkage'],
    tol=mejor['tol']
)

lda_best.fit(X_train, y_train_class) #se entrena con la mejor iteracion de los ciclos anidados y con la categorizacion que se hizo para poder usar lda
y_test_pred = lda_best.predict(X_test)
error_test = 1 - accuracy_score(y_test_class, y_pred_test)
print("Error en test:", round(error_test, 4))

In [ ]:
nuevo_dato = np.array([[2.5, 1.3, 4.2, 0.9]])  # Cambia según tus variables
prediccion = lda_best.predict(nuevo_dato)
print("Predicción para el nuevo dato:", prediccion)
